In [7]:
import os
import csv
import random
from tqdm import tqdm
import torch
from torch.nn.functional import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [9]:
# ===============================
# Config modèle Hugging Face
# ===============================
MODEL_ID = "matous-volf/political-leaning-politics"
TOKENIZER_ID = "launch/POLITICS"
MAX_TOKENS = 512  # Roberta limite
SAMPLE_CHARS = 10_000  # taille d'un chunk pour échantillonnage
N_SAMPLES = 10          # nombre d'échantillons par fichier

# Charger tokenizer + modèle
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.eval()  # mode inference

# Vérifier si CUDA dispo
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# ===============================
# Fonction pour découper texte en tokens
# ===============================
def chunk_text(text, max_tokens=MAX_TOKENS):
    tokens = tokenizer.encode(text, add_special_tokens=True)
    chunks = [tokens[i:i+max_tokens] for i in range(0, len(tokens), max_tokens)]
    return chunks

# ===============================
# Prédiction sur un chunk de tokens
# ===============================
def predict_chunk_tokens(tokens):
    input_ids = torch.tensor([tokens]).to(device)
    attention_mask = torch.ones_like(input_ids)
    with torch.no_grad():
        output = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = softmax(output.logits, dim=1)
    score, label_id = torch.max(probs, dim=1)
    label_map = {0: "left", 1: "center", 2: "right"}
    return label_map[label_id.item()], score.item()

# ===============================
# Fonction pour traiter un fichier énorme
# ===============================
def process_file_large(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    # 10 échantillons aléatoires de 100k caractères
    scores_total = {"left": 0.0, "center": 0.0, "right": 0.0}
    for _ in range(N_SAMPLES):
        if len(text) <= SAMPLE_CHARS:
            sample_text = text
        else:
            start = random.randint(0, len(text) - SAMPLE_CHARS)
            sample_text = text[start:start+SAMPLE_CHARS]
        
        chunks = chunk_text(sample_text)
        for chunk_tokens in chunks:
            label, score = predict_chunk_tokens(chunk_tokens)
            scores_total[label] += score
    
    # Moyenne sur tous les échantillons
    total_counts = N_SAMPLES * len(chunks)
    for k in scores_total:
        scores_total[k] /= total_counts

    final_label = max(scores_total, key=scores_total.get)
    final_score = scores_total[final_label]
    return final_label, final_score

# ===============================
# Traiter un dossier complet
# ===============================
def process_txt_folder(input_folder, output_csv="media_scores.csv"):
    txt_files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]
    results = []

    print(f"Processing {len(txt_files)} files...\n")
    for file_name in tqdm(txt_files):
        file_path = os.path.join(input_folder, file_name)
        label, score = process_file_large(file_path)
        results.append((file_name.replace(".txt",""), label, score))
        print(f"{file_name:30s} -> {label:6s} ({score:.2f})")
    
    # Sauvegarder CSV
    with open(output_csv, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["media", "predicted_label", "score"])
        writer.writerows(results)
    
    print(f"\n✅ Scores saved to {output_csv}")

# ===============================
# Exemple d'utilisation
# ===============================
if __name__ == "__main__":
    input_folder = "./data/medias_txt"  # dossier contenant tes fichiers TXT
    process_txt_folder(input_folder)


Processing 26 files...



  4%|▍         | 1/26 [00:32<13:31, 32.47s/it]

Axios.txt                      -> left   (0.61)


  8%|▊         | 2/26 [01:06<13:16, 33.17s/it]

Business Insider.txt           -> center (0.39)


...

Vox.txt                        -> left   (0.81)


 96%|█████████▌| 25/26 [12:00<00:24, 24.23s/it]

Washington Post.txt            -> left   (0.85)


100%|██████████| 26/26 [12:24<00:00, 28.64s/it]

Wired.txt                      -> left   (0.76)

✅ Scores saved to media_scores.csv
